# Sentiment Analysis / Emotion Modeling
### Klasifikasi Sentimen Ulasan Menggunakan SVM (Baseline)

Notebook ini menggunakan TF-IDF + SVM untuk memprediksi kelas sentimen (negative, neutral, positive) dari ulasan pelanggan.

In [ ]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC
from sklearn.metrics import classification_report
import joblib

In [ ]:
dataset_path = '../data/cleaned_reviews.csv'
if not os.path.exists(dataset_path):
    dataset_path = 'cleaned_reviews.csv'

df = pd.read_csv(dataset_path).dropna(subset=['text_clean'])

X_train, X_test, y_train, y_test = train_test_split(
    df['text_clean'], 
    df['sentiment'], 
    test_size=0.2, 
    random_state=42, 
    stratify=df['sentiment']
)

In [ ]:
vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

svm_model = SVC(kernel='linear', probability=True, random_state=42)
svm_model.fit(X_train_tfidf, y_train)

y_pred = svm_model.predict(X_test_tfidf)
print(classification_report(y_test, y_pred))

In [ ]:
# Tulis hasil prediksi ke CSV untuk agregasi
df['predicted_sentiment'] = svm_model.predict(vectorizer.transform(df['text_clean']))
df.to_csv('../data/reviews_with_sentiment.csv', index=False)

os.makedirs('../weights', exist_ok=True)
joblib.dump(svm_model, '../weights/svm_model.pkl')
print("Saved sentiment model.")